# Training Tasks Notebook

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from semantic_kernel.functions import kernel_function
from typing import Annotated
import random

class PokemonPlugin:
    @kernel_function(description="Provides a random pokemon name")
    def get_random_pokemon(self) -> Annotated[str, "Returns a random pokemon name"]:
        pokemon = ["Pikachu", "Charizard", "Bulbasaur"]

        return random.choice(pokemon)


In [3]:
from os import getenv
# Endpoint (LLM source)
from openai import AsyncOpenAI
# Service
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

client = AsyncOpenAI(
    api_key= getenv("GITHUB_TOKEN"),
    base_url= getenv("GITHUB_ENDPOINT")
)

service = OpenAIChatCompletion(
    ai_model_id= getenv("GITHUB_MODEL_ID"),
    async_client= client
)

In [4]:
from semantic_kernel.agents import ChatCompletionAgent

agent = ChatCompletionAgent(
    service= service,
    name= "PokemonAgent",
    instructions= """
    You are a helpful pokemon pokedex agent that gives descriptions of pokemon, if the user does not specify a pokemon name, describe a random pokemon.
    Put all sentences on a new line.
    """,
    plugins= [PokemonPlugin()]
)

In [5]:
user_inputs = [
    "Describe a pokemon", 
    "Describe the pokemon Squritle"
]

async def main():
    thread = None

    for prompt in user_inputs:
        print(f"\n\n--- User: {prompt}\n", end="", flush=True)
        first_response = True

        async for response in agent.invoke_stream(messages=prompt, thread=thread):
            thread = response.thread

            if first_response:
                first_response = False
                print(f"--- {response.name}: {response}\n", end="", flush=True)

            print(response, end="", flush=True)
    
    if thread:
        await thread.delete()

await main()



--- User: Describe a pokemon
--- PokemonAgent: 
Bulbasaur is a dual-type Grass/Poison Pokémon.

It is known as the Seed Pokémon.

Bulbasaur has a blue-green body with darker blue-green spots.

On its back, it carries a large plant bulb that will eventually bloom into a large flower.

Bulbasaur is often seen in grassy areas and is known for its gentle nature.

This Pokémon is capable of using a variety of moves, including its signature move, Solar Beam.

Bulbasaur evolves into Ivysaur starting at level 16.

--- User: Describe the pokemon Squritle
--- PokemonAgent: 
Squirtle is a Water-type Pokémon.

It is known as the Tiny Turtle Pokémon.

Squirtle has a small, light blue body and a shell that is a darker blue.

It is often characterized by its cute face and round eyes.

This Pokémon is known for its playful and friendly nature.

Squirtle is capable of using moves like Water Gun and Bubble.

Squirtle evolves into Wartortle starting at level 16.